# 论文 5：通过最小化描述长度保持神经网络简洁
## Hinton & Van Camp（1993）+ 现代剪枝技术

### 网络剪枝与压缩

核心观点：移除不必要的权重，可以得到更简单、泛化能力更强的网络。模型越小越好！

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 用于分类的简单神经网络

In [ ]:
def relu(x):
    return np.maximum(0, x)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

class SimpleNN:
    '简单的两层神经网络。'
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # 初始化权重
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.1
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.1
        self.b2 = np.zeros(output_dim)
        
        # 保存剪枝掩码
        self.mask1 = np.ones_like(self.W1)
        self.mask2 = np.ones_like(self.W2)
    
    def forward(self, X):
        '执行前向传播。'
        # 应用掩码，将已剪除的权重置零
        W1_masked = self.W1 * self.mask1
        W2_masked = self.W2 * self.mask2
        
        # 隐藏层
        self.h = relu(np.dot(X, W1_masked) + self.b1)
        
        # 输出层
        logits = np.dot(self.h, W2_masked) + self.b2
        probs = softmax(logits)
        
        return probs
    
    def predict(self, X):
        '预测类别标签'
        probs = self.forward(X)
        return np.argmax(probs, axis=1)
    
    def accuracy(self, X, y):
        '计算准确率。'
        predictions = self.predict(X)
        return np.mean(predictions == y)
    
    def count_parameters(self):
        '统计参数总数和仍然活跃（未被剪除）的参数数量。'
        total = self.W1.size + self.b1.size + self.W2.size + self.b2.size
        active = int(np.sum(self.mask1) + self.b1.size + np.sum(self.mask2) + self.b2.size)
        return total, active

# 测试网络
nn = SimpleNN(input_dim=10, hidden_dim=20, output_dim=3)
X_test = np.random.randn(5, 10)
y_test = nn.forward(X_test)
print(f"Network output shape: {y_test.shape}")
total, active = nn.count_parameters()
print(f"Parameters: {total} total, {active} active")

## 生成合成数据集

In [ ]:
def generate_classification_data(n_samples=1000, n_features=20, n_classes=3):
    """生成合成分类数据集。
    每个类别对应一个高斯簇。"""
    X = []
    y = []
    
    samples_per_class = n_samples // n_classes
    
    for c in range(n_classes):
        # 为当前类别随机生成中心
        center = np.random.randn(n_features) * 3
        
        # 围绕中心生成样本
        X_class = np.random.randn(samples_per_class, n_features) + center
        y_class = np.full(samples_per_class, c)
        
        X.append(X_class)
        y.append(y_class)
    
    X = np.vstack(X)
    y = np.concatenate(y)
    
    # 打乱样本顺序
    indices = np.random.permutation(len(X))
    X = X[indices]
    y = y[indices]
    
    return X, y

# 生成数据
X_train, y_train = generate_classification_data(n_samples=1000, n_features=20, n_classes=3)
X_test, y_test = generate_classification_data(n_samples=300, n_features=20, n_classes=3)

print(f"Training set: {X_train.shape}, {y_train.shape}")
print(f"Test set: {X_test.shape}, {y_test.shape}")
print(f"Class distribution: {np.bincount(y_train)}")

## 训练基线网络

In [ ]:
def train_network(model, X_train, y_train, X_test, y_test, epochs=100, lr=0.01):
    '简单的训练循环。'
    train_losses = []
    test_accuracies = []
    
    for epoch in range(epochs):
        # 前向传播
        probs = model.forward(X_train)
        
        # 交叉熵损失
        y_one_hot = np.zeros((len(y_train), model.output_dim))
        y_one_hot[np.arange(len(y_train)), y_train] = 1
        loss = -np.mean(np.sum(y_one_hot * np.log(probs + 1e-8), axis=1))
        
        # 反向传播（简化实现）
        batch_size = len(X_train)
        dL_dlogits = (probs - y_one_hot) / batch_size
        
        # W2、b2 的梯度
        dL_dW2 = np.dot(model.h.T, dL_dlogits)
        dL_db2 = np.sum(dL_dlogits, axis=0)
        
        # W1、b1 的梯度
        dL_dh = np.dot(dL_dlogits, (model.W2 * model.mask2).T)
        dL_dh[model.h <= 0] = 0  # ReLU 导数
        dL_dW1 = np.dot(X_train.T, dL_dh)
        dL_db1 = np.sum(dL_dh, axis=0)
        
        # 只更新掩码中仍处于活跃状态的权重
        model.W1 -= lr * dL_dW1 * model.mask1
        model.b1 -= lr * dL_db1
        model.W2 -= lr * dL_dW2 * model.mask2
        model.b2 -= lr * dL_db2
        
        # 跟踪指标
        train_losses.append(loss)
        test_acc = model.accuracy(X_test, y_test)
        test_accuracies.append(test_acc)
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}, Test Acc: {test_acc:.2%}")
    
    return train_losses, test_accuracies

# 训练基线模型
print("Training baseline network...\n")
baseline_model = SimpleNN(input_dim=20, hidden_dim=50, output_dim=3)
train_losses, test_accs = train_network(baseline_model, X_train, y_train, X_test, y_test, epochs=100)

baseline_acc = baseline_model.accuracy(X_test, y_test)
total_params, active_params = baseline_model.count_parameters()
print(f"\nBaseline: {baseline_acc:.2%} accuracy, {active_params} parameters")

## 基于幅值的剪枝

移除绝对值最小的权重。

In [ ]:
def prune_by_magnitude(model, pruning_rate):
    """剪除幅值最小的权重。
    
    pruning_rate：需要移除的权重比例（0～1）。"""
    # 收集所有权重
    all_weights = np.concatenate([model.W1.flatten(), model.W2.flatten()])
    all_magnitudes = np.abs(all_weights)
    
    # 查找阈值
    threshold = np.percentile(all_magnitudes, pruning_rate * 100)
    
    # 创建新的剪枝掩码
    model.mask1 = (np.abs(model.W1) > threshold).astype(float)
    model.mask2 = (np.abs(model.W2) > threshold).astype(float)
    
    print(f"Pruning threshold: {threshold:.6f}")
    print(f"Pruned {pruning_rate:.1%} of weights")
    
    total, active = model.count_parameters()
    print(f"Remaining parameters: {active}/{total} ({active/total:.1%})")

# 测试剪枝效果
import copy
pruned_model = copy.deepcopy(baseline_model)

print("Before pruning:")
acc_before = pruned_model.accuracy(X_test, y_test)
print(f"Accuracy: {acc_before:.2%}\n")

print("Pruning 50% of weights...")
prune_by_magnitude(pruned_model, pruning_rate=0.5)

print("\nAfter pruning (before retraining):")
acc_after = pruned_model.accuracy(X_test, y_test)
print(f"Accuracy: {acc_after:.2%}")
print(f"Accuracy drop: {(acc_before - acc_after):.2%}")

## 剪枝后微调

重新训练剩余权重，以恢复准确率。

In [ ]:
print("Fine-tuning pruned network...\n")
finetune_losses, finetune_accs = train_network(
    pruned_model, X_train, y_train, X_test, y_test, epochs=50, lr=0.005
)

acc_finetuned = pruned_model.accuracy(X_test, y_test)
total, active = pruned_model.count_parameters()

print(f"\n{'='*60}")
print("RESULTS:")
print(f"{'='*60}")
print(f"Baseline:     {baseline_acc:.2%} accuracy, {total_params} params")
print(f"Pruned 50%:   {acc_finetuned:.2%} accuracy, {active} params")
print(f"Compression:  {total_params/active:.1f}x smaller")
print(f"Acc. change:  {(acc_finetuned - baseline_acc):+.2%}")
print(f"{'='*60}")

## 迭代剪枝

逐步提高剪枝比例。

In [ ]:
def iterative_pruning(model, X_train, y_train, X_test, y_test, 
                     target_sparsity=0.9, num_iterations=5):
    '反复执行剪枝和微调。'
    results = []
    
    # 初始状态
    total, active = model.count_parameters()
    acc = model.accuracy(X_test, y_test)
    results.append({
        'iteration': 0,
        'sparsity': 0.0,
        'active_params': active,
        'accuracy': acc
    })
    
    # 逐渐增加稀疏度
    for i in range(num_iterations):
        # 本次迭代的稀疏性
        current_sparsity = target_sparsity * (i + 1) / num_iterations
        
        print(f"\nIteration {i+1}/{num_iterations}: Target sparsity {current_sparsity:.1%}")
        
        # 剪枝
        prune_by_magnitude(model, pruning_rate=current_sparsity)
        
        # 微调
        train_network(model, X_train, y_train, X_test, y_test, epochs=30, lr=0.005)
        
        # 记录结果
        total, active = model.count_parameters()
        acc = model.accuracy(X_test, y_test)
        results.append({
            'iteration': i + 1,
            'sparsity': current_sparsity,
            'active_params': active,
            'accuracy': acc
        })
    
    return results

# 运行迭代剪枝
iterative_model = copy.deepcopy(baseline_model)
results = iterative_pruning(iterative_model, X_train, y_train, X_test, y_test, 
                           target_sparsity=0.95, num_iterations=5)

## 可视化剪枝结果

In [ ]:
# 提取数据
sparsities = [r['sparsity'] for r in results]
accuracies = [r['accuracy'] for r in results]
active_params = [r['active_params'] for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 准确率与稀疏度
ax1.plot(sparsities, accuracies, 'o-', linewidth=2, markersize=10, color='steelblue')
ax1.axhline(y=baseline_acc, color='red', linestyle='--', linewidth=2, label='Baseline')
ax1.set_xlabel('Sparsity (Fraction Pruned)', fontsize=12)
ax1.set_ylabel('Test Accuracy', fontsize=12)
ax1.set_title('Accuracy vs Sparsity', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=11)
ax1.set_ylim([0, 1])

# 参数数量与准确率
ax2.plot(active_params, accuracies, 's-', linewidth=2, markersize=10, color='darkgreen')
ax2.axhline(y=baseline_acc, color='red', linestyle='--', linewidth=2, label='Baseline')
ax2.set_xlabel('Active Parameters', fontsize=12)
ax2.set_ylabel('Test Accuracy', fontsize=12)
ax2.set_title('Accuracy vs Model Size', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=11)
ax2.set_ylim([0, 1])
ax2.invert_xaxis()  # 右侧参数较少

plt.tight_layout()
plt.show()

print("\nKey observation: Can remove 90%+ of weights with minimal accuracy loss!")

## 可视化权重分布

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 基线权重
axes[0, 0].hist(baseline_model.W1.flatten(), bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 0].set_title('Baseline W1 Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Weight Value')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(baseline_model.W2.flatten(), bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 1].set_title('Baseline W2 Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Weight Value')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

# 剪枝后的权重（仅统计活跃权重）
pruned_W1 = iterative_model.W1[iterative_model.mask1 > 0]
pruned_W2 = iterative_model.W2[iterative_model.mask2 > 0]

axes[1, 0].hist(pruned_W1.flatten(), bins=50, color='darkgreen', alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Pruned W1 Distribution (Active Weights Only)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Weight Value')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(pruned_W2.flatten(), bins=50, color='darkgreen', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Pruned W2 Distribution (Active Weights Only)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Weight Value')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Pruned weights have larger magnitudes (small weights removed)")

## 可视化稀疏模式

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# W1 稀疏模式
im1 = ax1.imshow(iterative_model.mask1.T, cmap='RdYlGn', aspect='auto', interpolation='nearest')
ax1.set_xlabel('Input Dimension', fontsize=12)
ax1.set_ylabel('Hidden Dimension', fontsize=12)
ax1.set_title('W1 Sparsity Pattern (Green=Active, Red=Pruned)', fontsize=12, fontweight='bold')
plt.colorbar(im1, ax=ax1)

# W2 稀疏模式
im2 = ax2.imshow(iterative_model.mask2.T, cmap='RdYlGn', aspect='auto', interpolation='nearest')
ax2.set_xlabel('Hidden Dimension', fontsize=12)
ax2.set_ylabel('Output Dimension', fontsize=12)
ax2.set_title('W2 Sparsity Pattern (Green=Active, Red=Pruned)', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

total, active = iterative_model.count_parameters()
print(f"\nFinal sparsity: {(total - active) / total:.1%}")
print(f"Compression ratio: {total / active:.1f}x")

## MDL 原理

最小描述长度（Minimum Description Length）：模型越简单，泛化能力往往越好。

In [ ]:
def compute_mdl(model, X_train, y_train):
    """简化的 MDL 计算
    
    MDL = 模型成本 + 数据成本
    - 模型成本：编码权重所需的比特数
    - 数据成本：编码误差所需的比特数"""
    # 模型成本：参数数量（简化）
    total, active = model.count_parameters()
    model_cost = active  # 每个参数 = 1“位”（简化）
    
    # 数据成本：交叉熵损失
    probs = model.forward(X_train)
    y_one_hot = np.zeros((len(y_train), model.output_dim))
    y_one_hot[np.arange(len(y_train)), y_train] = 1
    data_cost = -np.sum(y_one_hot * np.log(probs + 1e-8))
    
    total_cost = model_cost + data_cost
    
    return {
        'model_cost': model_cost,
        'data_cost': data_cost,
        'total_cost': total_cost
    }

# 比较不同模型的 MDL
baseline_mdl = compute_mdl(baseline_model, X_train, y_train)
pruned_mdl = compute_mdl(iterative_model, X_train, y_train)

print("MDL Comparison:")
print(f"{'='*60}")
print(f"{'Model':<20} {'Model Cost':<15} {'Data Cost':<15} {'Total'}")
print(f"{'-'*60}")
print(f"{'Baseline':<20} {baseline_mdl['model_cost']:<15.0f} {baseline_mdl['data_cost']:<15.2f} {baseline_mdl['total_cost']:.2f}")
print(f"{'Pruned (95%)':<20} {pruned_mdl['model_cost']:<15.0f} {pruned_mdl['data_cost']:<15.2f} {pruned_mdl['total_cost']:.2f}")
print(f"{'='*60}")
print(f"\nPruned model has LOWER total cost → Better generalization!")

## 要点

### 神经网络剪枝

**核心思想**：移除不必要的权重，构建更简单、更小的网络。

### 基于幅值的剪枝

1. 按常规方式**训练**网络
2. **识别**幅值较小的权重：$|w| < \text{threshold}$
3. **移除**这些权重（将其置为 0，并通过掩码屏蔽）
4. **微调**剩余权重

### 迭代剪枝

通常比一次性剪枝效果更好：
```
for iteration in 1..N:
    prune small fraction (e.g., 20%)
    finetune
```

让网络逐渐适应。

### 结果（典型）：

- **50% 稀疏度**：通常没有精度损失
- **90% 稀疏度**：轻微的精度损失 (<2%)
- **95%+ 稀疏度**：明显退化

现代网络（ResNet、Transformer）通常可以达到 **90～95% 的稀疏度**，同时只受到很小影响。

### MDL 原理

$$
\text{MDL} = \underbrace{L(\text{Model})}_\text{complexity} + \underbrace{L(\text{Data | Model})}_\text{errors}
$$

**奥卡姆剃刀**：能够拟合数据的最简单解释（最小网络）通常是最好的。

### 剪枝的好处

1. **较小的模型**：内存更少，推理速度更快
2. **更好的泛化能力**：移除容易导致过拟合的参数
3. **更高的能源效率**：需要执行的运算更少
4. **可解释性**：结构更简单

### 剪枝类型

| 类型 | 移除对象 | 加速效果 |
|------|----------------|----------|
| **非结构化剪枝** | 单个权重 | 低（依赖稀疏运算） |
| **结构化剪枝** | 整个神经元或滤波器 | 高（仍可使用稠密运算） |
| **通道剪枝** | 整个通道 | 高 |
| **层剪枝** | 整层网络 | 非常高 |

### 现代技术：

1. **彩票假设**：
   - 剪枝后的网络可以从初始权重重新训练
   - “中奖彩票”存在于随机初始化中

2. **动态稀疏训练**：
   - 在训练过程中进行剪枝，而不是训练结束后再剪枝
   - 重新增长连接

3. **幅度+梯度**：
   - 使用梯度信息，而不仅仅是幅度
   - 去除小幅度和小梯度的权重

4. **可学习的稀疏性**：
   - L0/L1 正则化
   - 自动稀疏发现

### 实用技巧：

1. **从较小比例开始，逐步剪枝**：不要立即剪除 90% 的权重
2. **剪枝后进行微调**：这是恢复性能的关键
3. **逐层剪枝率**：不同层有不同的冗余度
4. **需要实际加速时采用结构化剪枝**：非结构化剪枝通常需要特殊硬件支持

### 何时适合剪枝

✅ **适合**：
- 部署（边缘设备、移动设备）
- 降低推理成本
- 模型压缩

❌ **不适合**：
- 非常小的模型（已经很高效）
- 训练加速（仅限结构化剪枝）

### 实践中的压缩率：

- **AlexNet**：9 倍压缩（无精度损失）
- **VGG-16**：13 倍压缩
- **ResNet-50**：5-7 倍压缩
- **BERT**：10-40 倍压缩（带量化）

### 关键见解：

**神经网络严重过度参数化！**

大多数权重对最终性能的贡献很小。剪枝可以揭示真正发挥作用的“核心”网络。

**“最好的模型是适合数据的最简单的模型”** - MDL 原理